# Sentiment Model Training

This mirrors `app/services/nlp_service.py` step by step, so you can inspect the data and metrics interactively before trusting the packaged script.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from app.services.nlp_service import clean_text

df = pd.read_csv('../data/reviews.csv')
df['clean_review'] = df['review'].apply(clean_text)
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'], test_size=0.25, random_state=42, stratify=df['sentiment'])

vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_v = vec.fit_transform(X_train)
X_test_v = vec.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_v, y_train)

print(classification_report(y_test, clf.predict(X_test_v)))
print(confusion_matrix(y_test, clf.predict(X_test_v)))

## Upgrade option: DistilBERT fine-tune (stretch goal)
Swap the TF-IDF + LogisticRegression above for a fine-tuned transformer if you want higher accuracy:

In [ ]:
# pip install transformers datasets
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
# model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
# ... tokenize df['clean_review'], wrap in a Dataset, then Trainer(...).train()
print('See commented cell above for the DistilBERT fine-tuning outline.')

In [ ]:
import joblib
joblib.dump(clf, '../app/models/sentiment_model.pkl')
joblib.dump(vec, '../app/models/vectorizer.pkl')
print('Saved.')